# TP3 — Étape 1

## Partie 1 : Classification binaire

### 1. Choix du jeu de données

On utilise **SMS Spam Collection** : chaque SMS est étiqueté `ham` (message normal) ou `spam` (message indésirable).

Le fichier `SMSSpamCollection` se trouve dans le même dossier que ce notebook. Source : [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).

In [1]:
import pandas as pd

In [2]:
import csv

# Les guillemets font partie du texte des SMS.
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    names=["label", "text"],
    quoting=csv.QUOTE_NONE
)

print(df.head())
print("Taille du dataset :", df.shape)
print(df["label"].value_counts())

  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
Taille du dataset : (5574, 2)
label
ham     4827
spam     747
Name: count, dtype: int64


In [3]:
df["target"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

print(df[["label", "target"]].head())

  label  target
0   ham       0
1   ham       0
2  spam       1
3   ham       0
4   ham       0


### 2. Prétraitement du texte

On reprend les étapes de TP1 en les adaptant aux SMS anglais : passage en minuscules, suppression de ponctuation, tokenisation, suppression des stopwords et lemmatisation.

In [4]:
import nltk

# Télécharger uniquement les ressources manquantes.
for resource, resource_path in [
    ("punkt_tab", "tokenizers/punkt_tab/english/"),
    ("stopwords", "corpora/stopwords"),
    ("wordnet", "corpora/wordnet.zip")
]:
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(resource, raise_on_error=True)

In [5]:
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [6]:
def clean_text(text):
    # Passage en minuscules
    text = text.lower()

    # Suppression de la ponctuation et des caractères spéciaux
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Tokenisation
    tokens = word_tokenize(text)

    # Suppression des stopwords anglais
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatisation
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [7]:
df["clean_text"] = df["text"].apply(clean_text)

print(df[["text", "clean_text"]].head())

                                                text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry 2 wkly comp win fa cup final tkts 2...  
3                u dun say early hor u c already say  
4                nah think go usf life around though  


In [8]:
X = df["clean_text"]
y = df["target"]

### 3. Séparation des données — 80 % / 20 %

On réserve 80 % des SMS à l'entraînement et 20 % au test. La stratification conserve approximativement les proportions de `ham` et de `spam`.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [10]:
print("Train :", len(X_train))
print("Test :", len(X_test))

Train : 4459
Test : 1115


### 4. Vectorisation du texte — TF-IDF

`TfidfVectorizer` transforme les textes en vecteurs numériques. On l'intègre dans un `Pipeline` avec chaque classifieur : pendant la validation croisée, le vocabulaire et les poids TF-IDF sont appris uniquement sur le pli d'entraînement.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

### 5. Entraînement de trois modèles

On reprend les trois algorithmes utilisés dans TP2 : régression logistique, SVM linéaire et Naive Bayes multinomial.

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}

### 6. Validation croisée — 5 plis

On évalue chaque modèle sur l'ensemble d'entraînement avec une validation stratifiée à **5 plis**. L'accuracy mesure la proportion de prédictions correctes. On affiche les cinq scores et leur moyenne, puis on entraîne chaque pipeline sur tout l'ensemble d'entraînement.

Les 20 % de test restent réservés pour la suite du TP.

In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [15]:
cv_results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print(name)
    print("Scores des 5 plis :", scores)
    print("Accuracy moyenne :", scores.mean())
    print()

    cv_results.append({
        "Modèle": name,
        "Accuracy CV moyenne": scores.mean()
    })

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

Logistic Regression
Scores des 5 plis : [0.96300448 0.95852018 0.96524664 0.95627803 0.9640853 ]
Accuracy moyenne : 0.9614269249545782



SVM
Scores des 5 plis : [0.98430493 0.98430493 0.98654709 0.98542601 0.98877666]
Accuracy moyenne : 0.9858719230169155



Naive Bayes
Scores des 5 plis : [0.96524664 0.96076233 0.96412556 0.96860987 0.9674523 ]
Accuracy moyenne : 0.9652393390808938



In [16]:
cv_results_df = pd.DataFrame(cv_results)
print(cv_results_df)

                Modèle  Accuracy CV moyenne
0  Logistic Regression             0.961427
1                  SVM             0.985872
2          Naive Bayes             0.965239
